# Job forcast

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import textwrap
import re
import shutil
import gc

# ============================================================
# ONE CODE ONLY: FORECAST PROJECTION OCCUPATION CHARTS
#
# This code reads:
#   /Users/YourUserName123/Downloads/Internship_SCIPE CI-SIP/MainProject/2_Job/forecast_projection_occupation.xlsx
#
# This code creates ONLY these 2 output folders in Downloads:
#   /Users/YourUserName123/Downloads/Forecast_Chart_1
#   /Users/YourUserName123/Downloads/Forecast_Table_1
#
# It does NOT edit the original Excel file.
# It does NOT create another Excel file.
# It does NOT create an output zip.
# ============================================================

# -------- INPUT EXCEL FILE --------
INPUT_FILE = (
    Path.home()
    / "Downloads"
    / "Internship_SCIPE CI-SIP"
    / "MainProject"
    / "2_Job"
    / "forecast_projection_occupation.xlsx"
)

# Backup if your computer really has /Downloads at the root
if not INPUT_FILE.exists():
    alt = Path("/Downloads/Internship_SCIPE CI-SIP/MainProject/2_Job/forecast_projection_occupation.xlsx")
    if alt.exists():
        INPUT_FILE = alt

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"File not found: {INPUT_FILE}")

# -------- OUTPUT FOLDERS IN DOWNLOADS --------
OUTPUT_ROOT = Path.home() / "Downloads"
CHART_DIR = OUTPUT_ROOT / "Forecast_Chart_1"
TABLE_DIR = OUTPUT_ROOT / "Forecast_Table_1"

# Clean old output folders first so you do not mix old and new files
for folder in [CHART_DIR, TABLE_DIR]:
    if folder.exists():
        shutil.rmtree(folder)
    folder.mkdir(parents=True, exist_ok=True)

print("INPUT FILE:", INPUT_FILE)
print("CHART FOLDER:", CHART_DIR)
print("TABLE FOLDER:", TABLE_DIR)
print("-" * 100)

# -------- Exact column names from this workbook --------
TITLE_COL = "2024 National Employment Matrix title"
EMP24 = "Employment, 2024"
EMP34 = "Employment, 2034"
CHG_NUM = "Employment change, numeric, 2024–34"
CHG_PCT = "Employment change, percent, 2024–34"
WAGE = "Median annual wage, dollars, 2024[1]"
OCC_TYPE = "Occupation type"
OPENINGS = "Occupational openings, 2024–34 annual average"
EDU = "Typical education needed for entry"

TEXT_KEYWORDS = [
    "title", "code", "type", "education", "experience", "training",
    "ooh", "content", "link", "factor", "category", "industry"
]


def clean_sheet(sheet_name):
    """Read and clean one Excel sheet."""
    print(f"Reading {sheet_name} ...", flush=True)
    df = pd.read_excel(INPUT_FILE, sheet_name=sheet_name, header=1)
    df = df.dropna(how="all").copy()
    df.columns = [str(c).strip() for c in df.columns]

    # Drop completely empty Unnamed columns
    empty_cols = [c for c in df.columns if c.lower().startswith("unnamed") and df[c].isna().all()]
    df = df.drop(columns=empty_cols)

    # Clean text columns
    for c in df.columns:
        if df[c].dtype == "object":
            df[c] = df[c].astype(str).str.strip()
            df[c] = df[c].replace({"nan": np.nan, "—": np.nan, "-": np.nan})

    # Convert numeric columns safely. Do not convert title/code/education text columns.
    for c in df.columns:
        c_lower = str(c).lower()
        is_text_col = any(k in c_lower for k in TEXT_KEYWORDS)
        if not is_text_col:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    return df


def clean_title(value):
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value).strip())


def remove_total(df, title_col):
    d = df.copy()
    d[title_col] = d[title_col].map(clean_title)
    mask = d[title_col].str.lower().ne("total, all occupations")
    mask &= d[title_col].ne("")
    return d[mask].copy()


def wrap_label(label, width=31):
    return "\n".join(textwrap.wrap(clean_title(label), width=width, break_long_words=False))


def save_table(df, rank, name):
    out = TABLE_DIR / f"{rank:02d}_{name}.csv"
    df.to_csv(out, index=False)
    return out


def finish_fig(fig, out):
    try:
        fig.tight_layout()
    except Exception:
        pass
    fig.savefig(out, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return out


def barh_chart(df, y_col, x_col, title, xlabel, rank, name,
               value_prefix="", value_suffix="", decimals=1):
    data = df[[y_col, x_col]].dropna(subset=[x_col]).copy()
    data[y_col] = data[y_col].map(clean_title)
    data = data[data[y_col].ne("")]

    # Sort so the most important value appears at the top.
    if (data[x_col] <= 0).all():
        data = data.sort_values(x_col, ascending=False)  # most negative on top
    else:
        data = data.sort_values(x_col, ascending=True)   # largest positive on top

    labels = [wrap_label(x, 34) for x in data[y_col]]
    values = data[x_col].astype(float).values
    height = min(14, max(5.5, 0.43 * len(data) + 2.2))

    fig, ax = plt.subplots(figsize=(12, height))
    ax.barh(labels, values)
    ax.set_title(title, fontsize=15, weight="bold", pad=14)
    ax.set_xlabel(xlabel)
    ax.grid(axis="x", alpha=0.25)
    ax.set_axisbelow(True)

    xmin, xmax = ax.get_xlim()
    span = xmax - xmin if xmax != xmin else 1
    for i, v in enumerate(values):
        if abs(v) >= 100:
            txt = f"{value_prefix}{v:,.0f}{value_suffix}"
        else:
            txt = f"{value_prefix}{v:,.{decimals}f}{value_suffix}"
        if v >= 0:
            ax.text(v + span * 0.01, i, txt, va="center", fontsize=9)
        else:
            ax.text(v - span * 0.01, i, txt, va="center", ha="right", fontsize=9)

    return finish_fig(fig, CHART_DIR / f"{rank:02d}_{name}.png")


def grouped_bar_chart(df, cat_col, value_cols, title, ylabel, rank, name, top_n=None):
    data = df[[cat_col] + value_cols].dropna(subset=value_cols, how="all").copy()
    data[cat_col] = data[cat_col].map(clean_title)
    data = data[data[cat_col].ne("")]

    if top_n is not None and len(data) > top_n:
        data = data.sort_values(value_cols[-1], ascending=False).head(top_n)

    x = np.arange(len(data))
    width = 0.38 if len(value_cols) == 2 else 0.25
    fig_w = max(12, min(18, len(data) * 0.8))

    fig, ax = plt.subplots(figsize=(fig_w, 7))
    for idx, col in enumerate(value_cols):
        offset = (idx - (len(value_cols) - 1) / 2) * width
        ax.bar(x + offset, data[col], width, label=col)

    ax.set_title(title, fontsize=15, weight="bold", pad=14)
    ax.set_ylabel(ylabel)
    ax.set_xticks(x)
    ax.set_xticklabels([wrap_label(v, 18) for v in data[cat_col]], rotation=45, ha="right")
    ax.grid(axis="y", alpha=0.25)
    ax.legend()
    ax.set_axisbelow(True)

    return finish_fig(fig, CHART_DIR / f"{rank:02d}_{name}.png")


def scatter_chart(df, x_col, y_col, size_col, label_col, title, xlabel, ylabel, rank, name, top_label_n=10):
    data = df[[x_col, y_col, size_col, label_col]].dropna().copy()
    data = data[(data[y_col] > 0) & (data[x_col] > 0)]
    if data.empty:
        return None

    size = np.sqrt(data[size_col].clip(lower=0).fillna(0) + 1) * 18
    fig, ax = plt.subplots(figsize=(11, 7))
    ax.scatter(data[x_col], data[y_col], s=size, alpha=0.65)
    ax.set_title(title, fontsize=15, weight="bold", pad=14)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.25)

    # Label only a few strongest points so the chart is not messy.
    label_data = data.sort_values([x_col, y_col], ascending=False).head(top_label_n)
    for _, row in label_data.iterrows():
        ax.annotate(clean_title(row[label_col]), (row[x_col], row[y_col]),
                    fontsize=8, xytext=(5, 5), textcoords="offset points")

    return finish_fig(fig, CHART_DIR / f"{rank:02d}_{name}.png")


# ============================================================
# READ NEEDED SHEETS ONLY
# ============================================================
t11 = remove_total(clean_sheet("Table 1.1"), TITLE_COL)
t12 = clean_sheet("Table 1.2")
t12[TITLE_COL] = t12[TITLE_COL].map(clean_title)
t12_line = t12[t12[OCC_TYPE].eq("Line item")].copy()

t13 = remove_total(clean_sheet("Table 1.3"), TITLE_COL)
t14 = remove_total(clean_sheet("Table 1.4"), TITLE_COL)
t15 = remove_total(clean_sheet("Table 1.5"), TITLE_COL)
t16 = remove_total(clean_sheet("Table 1.6"), TITLE_COL)

t110 = clean_sheet("Table 1.10")
t110[TITLE_COL] = t110[TITLE_COL].map(clean_title)
t110_line = t110[t110[OCC_TYPE].eq("Line item")].copy()

t111 = clean_sheet("Table 1.11")
t112 = clean_sheet("Table 1.12")

outputs = []

# 01 Major occupational groups by employment change
d1 = t11.sort_values(CHG_NUM, ascending=False)
outputs.append(save_table(d1, 1, "Major_Groups_Employment_Change_2024_2034"))
outputs.append(barh_chart(
    d1, TITLE_COL, CHG_NUM,
    "Major occupational groups by projected employment change, 2024–2034",
    "Employment change (thousands)",
    1, "Major_Groups_Employment_Change_2024_2034"
))

# 02 Major occupational groups by percent change
d2 = t11.sort_values(CHG_PCT, ascending=False)
outputs.append(save_table(d2, 2, "Major_Groups_Percent_Change_2024_2034"))
outputs.append(barh_chart(
    d2, TITLE_COL, CHG_PCT,
    "Major occupational groups by projected percent change, 2024–2034",
    "Employment change (%)",
    2, "Major_Groups_Percent_Change_2024_2034",
    value_suffix="%"
))

# 03 Top major groups, 2024 vs 2034
d3 = t11.sort_values(EMP34, ascending=False).head(12)
outputs.append(save_table(d3, 3, "Top_Major_Groups_Employment_2024_vs_2034"))
outputs.append(grouped_bar_chart(
    d3, TITLE_COL, [EMP24, EMP34],
    "Top major occupational groups: employment in 2024 vs projected 2034",
    "Employment (thousands)",
    3, "Top_Major_Groups_Employment_2024_vs_2034"
))

# 04 Fastest growing occupations
d4 = t13.sort_values(CHG_PCT, ascending=False).head(15)
outputs.append(save_table(d4, 4, "Fastest_Growing_Occupations_Percent"))
outputs.append(barh_chart(
    d4, TITLE_COL, CHG_PCT,
    "Fastest growing occupations by percent change, 2024–2034",
    "Employment change (%)",
    4, "Fastest_Growing_Occupations_Percent",
    value_suffix="%"
))

# 05 Occupations with most job growth
d5 = t14.sort_values(CHG_NUM, ascending=False).head(15)
outputs.append(save_table(d5, 5, "Occupations_With_Most_Job_Growth"))
outputs.append(barh_chart(
    d5, TITLE_COL, CHG_NUM,
    "Occupations with the most job growth, 2024–2034",
    "Employment change (thousands)",
    5, "Occupations_With_Most_Job_Growth"
))

# 06 Fastest declining occupations
d6 = t15.sort_values(CHG_PCT, ascending=True).head(15)
outputs.append(save_table(d6, 6, "Fastest_Declining_Occupations_Percent"))
outputs.append(barh_chart(
    d6, TITLE_COL, CHG_PCT,
    "Fastest declining occupations by percent change, 2024–2034",
    "Employment change (%)",
    6, "Fastest_Declining_Occupations_Percent",
    value_suffix="%"
))

# 07 Largest job declines
d7 = t16.sort_values(CHG_NUM, ascending=True).head(15)
outputs.append(save_table(d7, 7, "Occupations_With_Largest_Job_Declines"))
outputs.append(barh_chart(
    d7, TITLE_COL, CHG_NUM,
    "Occupations with the largest projected job declines, 2024–2034",
    "Employment change (thousands)",
    7, "Occupations_With_Largest_Job_Declines"
))

# 08 Top detailed occupations by annual openings
d8 = t110_line.sort_values(OPENINGS, ascending=False).head(20)
outputs.append(save_table(d8, 8, "Top_Detailed_Occupations_By_Annual_Openings"))
outputs.append(barh_chart(
    d8, TITLE_COL, OPENINGS,
    "Top detailed occupations by annual projected openings, 2024–2034",
    "Annual openings (thousands)",
    8, "Top_Detailed_Occupations_By_Annual_Openings"
))

# 09 Top detailed occupations by projected 2034 employment
d9 = t12_line.sort_values(EMP34, ascending=False).head(20)
outputs.append(save_table(d9, 9, "Top_Detailed_Occupations_By_Projected_2034_Employment"))
outputs.append(barh_chart(
    d9, TITLE_COL, EMP34,
    "Top detailed occupations by projected 2034 employment",
    "Employment in 2034 (thousands)",
    9, "Top_Detailed_Occupations_By_Projected_2034_Employment"
))

# 10 Highest-paying detailed occupations
d10 = t12_line.dropna(subset=[WAGE]).sort_values(WAGE, ascending=False).head(20)
outputs.append(save_table(d10, 10, "Highest_Paying_Detailed_Occupations"))
outputs.append(barh_chart(
    d10, TITLE_COL, WAGE,
    "Highest-paying detailed occupations, 2024 median annual wage",
    "Median annual wage ($)",
    10, "Highest_Paying_Detailed_Occupations",
    value_prefix="$",
    decimals=0
))

# 11 Detailed occupation count by education
d11 = (
    t12_line.dropna(subset=[EDU])
    .groupby(EDU, dropna=True)
    .size()
    .reset_index(name="Occupation count")
    .sort_values("Occupation count", ascending=False)
)
outputs.append(save_table(d11, 11, "Detailed_Occupation_Count_By_Education"))
outputs.append(barh_chart(
    d11, EDU, "Occupation count",
    "Detailed occupation count by typical entry education",
    "Number of detailed occupations",
    11, "Detailed_Occupation_Count_By_Education",
    decimals=0
))

# 12 Median wage by education
d12 = (
    t12_line.dropna(subset=[EDU, WAGE])
    .groupby(EDU, dropna=True)[WAGE]
    .median()
    .reset_index(name="Median wage")
    .sort_values("Median wage", ascending=False)
)
outputs.append(save_table(d12, 12, "Median_Wage_By_Education"))
outputs.append(barh_chart(
    d12, EDU, "Median wage",
    "Median wage by typical entry education",
    "Median annual wage ($)",
    12, "Median_Wage_By_Education",
    value_prefix="$",
    decimals=0
))

# 13 STEM vs non-STEM
cat_col = "Occupation category"
d13 = t111.dropna(subset=[EMP24, EMP34, CHG_PCT]).copy()
d13[cat_col] = d13[cat_col].map(clean_title)
d13 = d13[~d13[cat_col].str.lower().eq("total, all occupations")]
outputs.append(save_table(d13, 13, "STEM_vs_Non_STEM_Employment_Change"))
outputs.append(grouped_bar_chart(
    d13, cat_col, [EMP24, EMP34],
    "STEM vs non-STEM employment: 2024 vs projected 2034",
    "Employment (thousands)",
    13, "STEM_vs_Non_STEM_Employment_2024_vs_2034"
))

# 14 Growth percent vs median wage scatter
d14 = t12_line.dropna(subset=[CHG_PCT, WAGE, CHG_NUM]).copy()
d14 = d14[(d14[CHG_PCT] > 0) & (d14[WAGE] > 0)]
outputs.append(save_table(
    d14.sort_values([CHG_PCT, WAGE], ascending=False).head(50),
    14, "Growth_Percent_vs_Median_Wage_Detailed_Occupations"
))
scatter_out = scatter_chart(
    d14, CHG_PCT, WAGE, CHG_NUM, TITLE_COL,
    "Growth percent vs wage for detailed occupations",
    "Employment change (%)",
    "Median annual wage ($)",
    14, "Growth_Percent_vs_Median_Wage_Detailed_Occupations"
)
if scatter_out is not None:
    outputs.append(scatter_out)

# 15 Utilization factor categories from Table 1.12
factor_col = "Factors affecting occupational utilization"
d15_raw = t112.dropna(subset=[factor_col]).copy()


def factor_category(text):
    t = str(text).lower()
    if "demand change" in t and "share increases" in t:
        return "Demand change: share increases"
    if "demand change" in t and "share decreases" in t:
        return "Demand change: share decreases"
    if "productivity change" in t and "share increases" in t:
        return "Productivity change: share increases"
    if "productivity change" in t and "share decreases" in t:
        return "Productivity change: share decreases"
    if "restructuring" in t and "share increases" in t:
        return "Restructuring: share increases"
    if "restructuring" in t and "share decreases" in t:
        return "Restructuring: share decreases"
    return "Other"


d15_raw["Factor category"] = d15_raw[factor_col].map(factor_category)
d15 = d15_raw.groupby("Factor category").size().reset_index(name="Count").sort_values("Count", ascending=False)
outputs.append(save_table(d15, 15, "Utilization_Factor_Category_Counts"))
outputs.append(barh_chart(
    d15, "Factor category", "Count",
    "Factors affecting occupational utilization, 2024–2034",
    "Number of factor notes",
    15, "Utilization_Factor_Category_Counts",
    decimals=0
))

print("-" * 100)
print("DONE")
print("Charts created:", len(list(CHART_DIR.glob("*.png"))))
print("Tables created:", len(list(TABLE_DIR.glob("*.csv"))))
print("Saved chart folder:", CHART_DIR)
print("Saved table folder:", TABLE_DIR)
print("\nEXPECTED OUTPUT:")
print("/Users/YourUserName123/Downloads/Forecast_Chart_1")
print("/Users/YourUserName123/Downloads/Forecast_Table_1")

gc.collect()
